# Substation Shifting Proportionality Test

**Question:** Is the per-household shift rate the same across all substations, or do some substations differ beyond random noise?

---

## Why not Pearson correlation on totals?

Correlating *total shift* with *n_households* (r ≈ 0.995) is nearly tautological:
`total_shift = per_capita_rate × n_households`, so the 15:1 range in population size (297–4,536 HHs) dominates the correlation completely. This confirms that bigger substations shift more aggregate load — which is obvious — not that the *rate* is proportional.

## Why not chi-square?

Chi-square tests independence between categorical variables. Both shifting and population are continuous ratio-scale quantities; binning them is arbitrary and lossy.

## Correct approach: household-level one-way ANOVA

We use each individual household's per-day shift as the unit of observation (n ≈ 28,195), with **substation as the grouping factor (21 levels)**:

- **H₀:** All substations have the same mean per-household shift rate — population size is the only predictor.
- **H₁:** At least one substation has a systematically different per-household rate.
- **F-ratio** = between-substation variance / within-substation variance.
- **η²** (eta-squared) = SS_between / SS_total — fraction of shift variance explained by substation membership.

This is valid because we have true replication: hundreds to thousands of households per substation. ANOVA on the *substation* level (n = 1 per group) would be meaningless — ANOVA on the *household* level is exactly right.

---

**Red day only:** 2014-07-03 (only red day in the dataset; strongest Tempo price signal).  
**TOU tariff window — hours 13–19:** Hours where TOU strictly suppresses load (verified by inspection: 0 % of households show increases in h13–19, 8.9 % show increases only in h20–24 = reallocated demand).

In [1]:
import os
import zipfile
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

BASE_DIR    = '/home/user/capstone_visuals'
ZIP_PATH    = os.path.join(BASE_DIR, 'dist_net.zip')
CONTROL_CSV = os.path.join(BASE_DIR, 'data', 'control_profile.csv')
TEMPO_CSV   = os.path.join(BASE_DIR, 'data', 'tempo_shifted_profile.csv')
TOU_ZIP     = os.path.join(BASE_DIR, 'data', 'tou_shifted_profile1.zip')  # updated dataset
TOU_CSV     = 'tou_shifted_profile1.csv'
EXTRACT_DIR = '/tmp/dist_net'
OUTPUT_DIR  = os.path.join(BASE_DIR, 'output')

RED_DAY       = '2014-07-03'
HOUR_COLS     = [str(i) for i in range(1, 25)]   # all 24 hours — Tempo
TOU_HOUR_COLS = [str(i) for i in range(13, 20)]  # TOU tariff window: hours 13-19

def normalize_hid(x):
    try:    return str(int(float(x)))
    except: return None

print('Imports OK')
print('Red day:      ', RED_DAY)
print('Tempo hours:  ', f'{HOUR_COLS[0]}-{HOUR_COLS[-1]}  ({len(HOUR_COLS)} hours)')
print('TOU window:   ', f'{TOU_HOUR_COLS[0]}-{TOU_HOUR_COLS[-1]}  ({len(TOU_HOUR_COLS)} hours)')
print('TOU dataset:  ', os.path.basename(TOU_ZIP))

Imports OK
Red day:       2014-07-03
Tempo hours:   1-24  (24 hours)
TOU window:    13-19  (7 hours)
TOU dataset:   tou_shifted_profile1.zip


## 1. Build HID → Substation Mapping from Shapefiles

In [2]:
# Extract network zip if needed
flag = os.path.join(EXTRACT_DIR, '.done')
if not os.path.exists(flag):
    print('Extracting dist_net.zip...')
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
    open(flag, 'w').close()

# Load all H-nodes across substations to build HID → substation map
content = os.path.join(EXTRACT_DIR, 'content', 'output')
all_nodes = []

for folder in sorted(os.listdir(content)):
    fp = os.path.join(content, folder, f'{folder}-nodelist-HID.shp')
    if not os.path.exists(fp):
        continue
    gdf = gpd.read_file(fp)
    gdf['substation'] = folder
    all_nodes.append(gdf)

nodes = pd.concat(all_nodes, ignore_index=True)
h_nodes = nodes[nodes['label'] == 'H'].copy()
h_nodes['hid_key'] = h_nodes['hid'].apply(normalize_hid)
h_nodes = h_nodes.dropna(subset=['hid_key'])

# Household count per substation
hh_counts = h_nodes.groupby('substation')['hid_key'].nunique().rename('n_households')

print(f'Total H-nodes: {len(h_nodes):,}')
print(f'Substations:   {hh_counts.shape[0]}')
print()
print(hh_counts.sort_values(ascending=False).to_string())

Total H-nodes: 28,221
Substations:   21

substation
139356    4536
121578    3719
121579    3146
121581    2428
121582    1780
121575    1277
147948    1184
121577    1098
156947     994
121587     978
121586     906
147966     906
147965     904
147947     822
121580     807
144353     729
121583     511
121584     455
139355     370
156257     370
147963     297


## 2. Load CSVs, Match Households, Compute Per-Household Shifts

For each household matched across control ↔ scenario on the red day:
- **Tempo shift** = sum(tempo_h1-24) − sum(ctrl_h1-24)
- **TOU shift** = sum(ctrl_h13-19) − sum(tou_h13-19)  (load shed; always ≥ 0 in tariff window)

In [3]:
# Load and filter to red day
print('Loading control...')
ctrl = pd.read_csv(CONTROL_CSV)
ctrl['hid_key'] = ctrl['hid'].apply(normalize_hid)
ctrl = ctrl[ctrl['date'] == RED_DAY]

print('Loading tempo...')
tempo = pd.read_csv(TEMPO_CSV)
tempo['hid_key'] = tempo['hid'].apply(normalize_hid)
tempo = tempo[tempo['date'] == RED_DAY]

print(f'Loading TOU ({TOU_CSV})...')
with zipfile.ZipFile(TOU_ZIP) as z:
    with z.open(TOU_CSV) as f:
        tou = pd.read_csv(f)
tou['hid_key'] = tou['hid'].apply(normalize_hid)
tou = tou[tou['date'] == RED_DAY]

print(f'\nControl  — red day rows: {len(ctrl):,}   unique HIDs: {ctrl["hid_key"].nunique():,}')
print(f'Tempo    — red day rows: {len(tempo):,}   unique HIDs: {tempo["hid_key"].nunique():,}')
print(f'TOU      — red day rows: {len(tou):,}   unique HIDs: {tou["hid_key"].nunique():,}')

Loading control...


Loading tempo...


Loading TOU (tou_shifted_profile1.csv)...



Control  — red day rows: 28,195   unique HIDs: 28,195
Tempo    — red day rows: 28,195   unique HIDs: 28,195
TOU      — red day rows: 28,195   unique HIDs: 28,195


In [4]:
# Add substation labels
hid_sub = h_nodes[['hid_key', 'substation']].drop_duplicates(subset='hid_key')

def add_substation(df):
    return df.merge(hid_sub, on='hid_key', how='inner')

ctrl_s  = add_substation(ctrl)
tempo_s = add_substation(tempo)
tou_s   = add_substation(tou)

# Build household-level shift table (one row per household)
# Tempo: total 24h shift per household
ctrl_24h  = ctrl_s [['hid_key','substation'] + HOUR_COLS].copy()
tempo_24h = tempo_s[['hid_key'] + HOUR_COLS].copy()
ctrl_24h ['ctrl_24h']  = ctrl_24h [HOUR_COLS].sum(axis=1)
tempo_24h['tempo_24h'] = tempo_24h[HOUR_COLS].sum(axis=1)
hh_tempo = ctrl_24h[['hid_key','substation','ctrl_24h']].merge(
    tempo_24h[['hid_key','tempo_24h']], on='hid_key')
hh_tempo['shift_tempo'] = hh_tempo['tempo_24h'] - hh_tempo['ctrl_24h']

# TOU: tariff-window (h13-19) load shed per household
ctrl_win = ctrl_s [['hid_key','substation'] + TOU_HOUR_COLS].copy()
tou_win  = tou_s  [['hid_key'] + TOU_HOUR_COLS].copy()
ctrl_win ['ctrl_win'] = ctrl_win [TOU_HOUR_COLS].sum(axis=1)
tou_win  ['tou_win']  = tou_win  [TOU_HOUR_COLS].sum(axis=1)
hh_tou = ctrl_win[['hid_key','substation','ctrl_win']].merge(
    tou_win[['hid_key','tou_win']], on='hid_key')
hh_tou['shift_tou'] = hh_tou['ctrl_win'] - hh_tou['tou_win']  # load shed >= 0

print(f'Household-level Tempo rows: {len(hh_tempo):,}')
print(f'Household-level TOU rows:   {len(hh_tou):,}')
print()
print('Households per substation:')
print(hh_tempo.groupby('substation').size().sort_values(ascending=False).to_string())

Household-level Tempo rows: 28,195
Household-level TOU rows:   28,195

Households per substation:
substation
139356    4529
121578    3719
121579    3145
121581    2426
121582    1779
121575    1275
147948    1184
121577    1096
156947     991
121587     977
121586     906
147966     906
147965     901
147947     822
121580     807
144353     729
121583     511
121584     455
139355     370
156257     370
147963     297


In [5]:
# Per-substation summary: mean per-household rate and 95% CI
def substation_summary(hh_df, shift_col):
    grp = hh_df.groupby('substation')[shift_col]
    summary = grp.agg(n='count', mean='mean', std='std').reset_index()
    summary['se']    = summary['std'] / np.sqrt(summary['n'])
    summary['ci95']  = 1.96 * summary['se']
    return summary.set_index('substation').sort_values('n', ascending=False)

sub_tempo = substation_summary(hh_tempo, 'shift_tempo')
sub_tou   = substation_summary(hh_tou,   'shift_tou')

print('=== Per-substation mean shift rate (kWh/household) ===')
print()
print('TEMPO (24h net shift):')
print(sub_tempo[['n','mean','std','se','ci95']].round(5).to_string())
print()
print('TOU (h13-19 load shed):')
print(sub_tou[['n','mean','std','se','ci95']].round(5).to_string())
print()
print(f'Tempo  — overall CV: {100*sub_tempo["mean"].std()/sub_tempo["mean"].abs().mean():.1f}%')
print(f'TOU    — overall CV: {100*sub_tou["mean"].std()/sub_tou["mean"].mean():.1f}%')

=== Per-substation mean shift rate (kWh/household) ===

TEMPO (24h net shift):
               n     mean      std       se     ci95
substation                                          
139356      4529 -0.49452  1.62939  0.02421  0.04745
121578      3719 -0.47257  1.61952  0.02656  0.05205
121579      3145 -0.45824  1.55933  0.02781  0.05450
121581      2426 -0.52076  1.68716  0.03425  0.06714
121582      1779 -0.55727  1.80115  0.04270  0.08370
121575      1275 -0.53162  1.78394  0.04996  0.09792
147948      1184 -0.50242  1.71284  0.04978  0.09757
121577      1096 -0.53213  1.80384  0.05449  0.10679
156947       991 -0.43393  1.51333  0.04807  0.09422
121587       977 -0.47694  1.66861  0.05338  0.10463
121586       906 -0.57919  1.83302  0.06090  0.11936
147966       906 -0.57624  1.75948  0.05845  0.11457
147965       901 -0.48475  1.65661  0.05519  0.10817
147947       822 -0.52073  1.74319  0.06080  0.11917
121580       807 -0.43775  1.49117  0.05249  0.10288
144353       729 -0.

## 3. One-Way ANOVA: Does Substation Explain Per-Household Shift Rate?

H₀: All substation group means are equal (substation membership adds no explanatory power beyond noise).  
H₁: At least one substation has a different mean per-household shift rate.

**η²** = SS_between / SS_total — proportion of total variance in household-level shifts explained by substation.  
**ω²** = (SS_between − (k−1)·MS_within) / (SS_total + MS_within) — bias-corrected version of η².

In [6]:
def one_way_anova(hh_df, shift_col, scenario_label):
    groups = [grp[shift_col].values for _, grp in hh_df.groupby('substation')]
    k  = len(groups)                      # number of substations
    N  = sum(len(g) for g in groups)      # total households
    grand_mean = hh_df[shift_col].mean()

    # Sums of squares
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_within  = sum(((g - g.mean())**2).sum() for g in groups)
    ss_total   = ss_between + ss_within

    df_between = k - 1
    df_within  = N - k
    ms_between = ss_between / df_between
    ms_within  = ss_within  / df_within

    F = ms_between / ms_within
    p = stats.f.sf(F, df_between, df_within)

    # Effect sizes
    eta2  = ss_between / ss_total
    omega2 = (ss_between - df_between * ms_within) / (ss_total + ms_within)

    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '(ns)'
    print(f'=== One-Way ANOVA: {scenario_label} ===')
    print(f'  k = {k} substations   N = {N:,} households')
    print(f'  F({df_between}, {df_within}) = {F:.4f}   p = {p:.2e}  {sig}')
    print(f'  η²  = {eta2:.4f}  ({100*eta2:.2f}% of variance explained by substation)')
    print(f'  ω²  = {omega2:.4f}  (bias-corrected)')
    print()
    return F, p, eta2, omega2

F_t, p_t, eta2_t, omega2_t = one_way_anova(hh_tempo, 'shift_tempo', 'TEMPO (24h net shift, red day)')
F_u, p_u, eta2_u, omega2_u = one_way_anova(hh_tou,   'shift_tou',   'TOU   (h13-19 load shed, red day)')

=== One-Way ANOVA: TEMPO (24h net shift, red day) ===
  k = 21 substations   N = 28,195 households
  F(20, 28174) = 0.8667   p = 6.31e-01  (ns)
  η²  = 0.0006  (0.06% of variance explained by substation)
  ω²  = -0.0001  (bias-corrected)

=== One-Way ANOVA: TOU   (h13-19 load shed, red day) ===
  k = 21 substations   N = 28,195 households
  F(20, 28174) = 8.5554   p = 6.50e-26  ***
  η²  = 0.0060  (0.60% of variance explained by substation)
  ω²  = 0.0053  (bias-corrected)



## 4. Visualisation: Per-Substation Mean Shift Rates with 95% CIs

Substations sorted by household count (largest → smallest). Overlapping CIs suggest rates are statistically indistinguishable; non-overlapping CIs identify substations that genuinely differ.

In [7]:
BG  = '#06090f';  TXT = '#c8d8e8';  DIM = '#2a3f55'
C_TEMPO = '#ff8c00';  C_TOU = '#00b4d8';  C_MEAN = '#ffffff'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'text.color': TXT, 'axes.labelcolor': TXT,
    'xtick.color': TXT, 'ytick.color': TXT,
    'axes.edgecolor': DIM, 'grid.color': DIM,
    'font.family': 'monospace', 'font.size': 9,
})

fig, axes = plt.subplots(1, 2, figsize=(15, 6), facecolor=BG)

for ax, sub_df, color, scenario, ylabel, F, p, eta2 in [
    (axes[0], sub_tempo, C_TEMPO, 'TEMPO', 'Mean shift per household (kWh, 24h)', F_t, p_t, eta2_t),
    (axes[1], sub_tou,   C_TOU,   'TOU',   'Mean load shed per household (kWh, h13-19)', F_u, p_u, eta2_u),
]:
    subs = sub_df.index.tolist()          # already sorted by n descending
    y    = np.arange(len(subs))
    means  = sub_df['mean'].values
    ci95   = sub_df['ci95'].values
    ns     = sub_df['n'].values
    grand  = np.average(means, weights=ns)

    ax.barh(y, means, color=color, alpha=0.5, height=0.6, zorder=3)
    ax.errorbar(means, y, xerr=ci95, fmt='none', color='white',
                elinewidth=0.9, capsize=3, capthick=0.9, zorder=5)
    ax.axvline(grand, color=C_MEAN, lw=1.2, linestyle='--', alpha=0.8,
               label=f'Weighted mean = {grand:.4f}')
    ax.axvline(0,     color=DIM,    lw=0.7, zorder=2)

    ax.set_yticks(y)
    ax.set_yticklabels([f'{s}  (n={n:,})' for s, n in zip(subs, ns)], fontsize=7)
    ax.set_xlabel(ylabel, fontsize=8)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    ax.set_title(f'{scenario}  |  F={F:.1f}  p{sig}  η²={eta2:.3f}\n'
                 'Error bars = 95% CI on household mean', fontsize=9)
    ax.grid(axis='x', lw=0.35, alpha=0.5)
    ax.legend(fontsize=7.5, facecolor='#0c1622', edgecolor=DIM, labelcolor=TXT)

fig.suptitle('Per-Household Shift Rates by Substation  |  Red Day (2014-07-03)',
             color='white', fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, '07_shifting_proportionality.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved to', out_path)

Saved to /home/user/capstone_visuals/output/07_shifting_proportionality.png


## 5. Summary

In [8]:
rows = []
for scenario, hh_df, shift_col, F, p, eta2, omega2 in [
    ('Tempo (24h net)',     hh_tempo, 'shift_tempo', F_t, p_t, eta2_t, omega2_t),
    ('TOU (h13-19 shed)',   hh_tou,   'shift_tou',   F_u, p_u, eta2_u, omega2_u),
]:
    grand_mean = hh_df[shift_col].mean()
    cv = 100 * hh_df.groupby('substation')[shift_col].mean().std() / abs(grand_mean)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    rows.append({
        'Scenario':       scenario,
        'N (households)': len(hh_df),
        'k (substations)': hh_df['substation'].nunique(),
        'Grand mean (kWh/HH)': round(grand_mean, 5),
        'CV of substation means (%)': round(cv, 1),
        'F-statistic': round(F, 2),
        'p-value': f'{p:.2e}',
        'Significant': sig,
        'eta2': round(eta2, 4),
        'omega2': round(omega2, 4),
    })

summary = pd.DataFrame(rows)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print(summary.T.to_string())

                                          0                  1
Scenario                    Tempo (24h net)  TOU (h13-19 shed)
N (households)                        28195              28195
k (substations)                          21                 21
Grand mean (kWh/HH)                -0.49892            0.13637
CV of substation means (%)              9.9               16.9
F-statistic                            0.87               8.56
p-value                            6.31e-01           6.50e-26
Significant                              ns                ***
eta2                                 0.0006              0.006
omega2                              -0.0001             0.0053


In [9]:
# Match households across control/tempo/tou for per-HH load comparisons
merged_tempo = ctrl_s[['hid_key'] + HOUR_COLS].merge(
    tempo_s[['hid_key'] + HOUR_COLS], on='hid_key', suffixes=('_c', '_t'))
merged_tou = ctrl_s[['hid_key'] + HOUR_COLS].merge(
    tou_s[['hid_key'] + HOUR_COLS], on='hid_key', suffixes=('_c', '_u'))

N = len(merged_tempo)

# Identify system peak hour (highest aggregate control load)
peak_hour = max(HOUR_COLS, key=lambda h: ctrl_s[h].sum())
print(f'System peak hour (control, red day): hour {peak_hour}:00')
print(f'  (Note: hour {peak_hour} is a TOU reallocation hour — outside tariff window h13-19)')
print()

# ── Helper ──────────────────────────────────────────────────────────────────
def report(label, ctrl_vals, scen_vals, n_hh):
    ctrl_hh = ctrl_vals.mean()
    scen_hh = scen_vals.mean()
    red_hh  = ctrl_hh - scen_hh
    pct     = 100 * red_hh / ctrl_hh
    print(f'  {label}')
    print(f'    Control:        {ctrl_hh:>10.4f} kWh/HH     {ctrl_hh*n_hh:>12.1f} kWh system-wide')
    print(f'    Scenario:       {scen_hh:>10.4f} kWh/HH     {scen_hh*n_hh:>12.1f} kWh system-wide')
    print(f'    Reduction:      {red_hh:>+10.4f} kWh/HH     {red_hh*n_hh:>+12.1f} kWh system-wide  ({pct:+.2f}%)')
    print()

# ── Peak hour ────────────────────────────────────────────────────────────────
print(f'=== At system peak hour {peak_hour}:00 ===')
report('TEMPO',
       merged_tempo[peak_hour + '_c'],
       merged_tempo[peak_hour + '_t'], N)
report('TOU  (reallocation hour — load INCREASES here)',
       merged_tou[peak_hour + '_c'],
       merged_tou[peak_hour + '_u'], N)

# ── Full period ───────────────────────────────────────────────────────────────
ctrl_24  = merged_tempo[[h + '_c' for h in HOUR_COLS]].sum(axis=1)
tempo_24 = merged_tempo[[h + '_t' for h in HOUR_COLS]].sum(axis=1)
ctrl_win = merged_tou  [[h + '_c' for h in TOU_HOUR_COLS]].sum(axis=1)
tou_win  = merged_tou  [[h + '_u' for h in TOU_HOUR_COLS]].sum(axis=1)

print('=== Full red day — Tempo (hours 1-24) ===')
report('TEMPO', ctrl_24, tempo_24, N)

print('=== TOU tariff window (hours 13-19) ===')
report('TOU', ctrl_win, tou_win, N)

# TOU: participating households only
tou_shed = ctrl_win - tou_win
n_part = (tou_shed > 1e-9).sum()
print(f'  Participating households: {n_part:,} / {N:,} ({100*n_part/N:.1f}%)')
print(f'    Avg reduction per participating HH: '
      f'{tou_shed[tou_shed > 1e-9].mean():+.4f} kWh  '
      f'({100*tou_shed[tou_shed>1e-9].mean()/ctrl_win.mean():+.2f}% of window baseline)')

System peak hour (control, red day): hour 22:00
  (Note: hour 22 is a TOU reallocation hour — outside tariff window h13-19)

=== At system peak hour 22:00 ===
  TEMPO
    Control:            0.7189 kWh/HH          20268.1 kWh system-wide
    Scenario:           0.6874 kWh/HH          19382.0 kWh system-wide
    Reduction:         +0.0314 kWh/HH           +886.2 kWh system-wide  (+4.37%)

  TOU  (reallocation hour — load INCREASES here)
    Control:            0.7189 kWh/HH          20268.1 kWh system-wide
    Scenario:           0.7506 kWh/HH          21162.0 kWh system-wide
    Reduction:         -0.0317 kWh/HH           -893.9 kWh system-wide  (-4.41%)

=== Full red day — Tempo (hours 1-24) ===
  TEMPO
    Control:           11.4875 kWh/HH         323890.6 kWh system-wide
    Scenario:          10.9886 kWh/HH         309823.4 kWh system-wide
    Reduction:         +0.4989 kWh/HH         +14067.1 kWh system-wide  (+4.34%)

=== TOU tariff window (hours 13-19) ===
  TOU
    Control:    

## 6. Average Reduction in Peak Load — by Household and System-Wide

**Peak hour** = hour with highest aggregate system load under control on the red day.  
Results reported for (a) the peak hour specifically and (b) the full relevant period (24h for Tempo; tariff window h13–19 for TOU).

> **Note on TOU and the system peak:** The system peak falls at hour 22 (10pm), which is *outside* the TOU tariff window and is one of the reallocation hours (h20–24). TOU therefore *increases* load at hour 22 — the demand suppressed in h13–19 arrives here. The correct "reduction" metric for TOU is the load shed within the tariff window itself.